In [1]:
import pandas as pd
import sklearn
print(pd.__version__)
print(sklearn.__version__)

2.3.3
1.6.1


In [2]:
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "label", "difficulty"
]

print(len(columns))


df_train = pd.read_csv("../data/raw/KDDTrain+.txt", header=None, names=columns)
df_test = pd.read_csv("../data/raw/KDDTest+.txt", header=None, names=columns)
print(f" Shape of training data: {df_train.shape}")
print(f" Shape of test data: {df_test.shape}")

43


 Shape of training data: (125973, 43)
 Shape of test data: (22544, 43)


## Isolation Forest baseline

Isolation Forest flags points that are easy to separate from the rest of the data. It is trained here two ways:

1. **Mixed training** — on all training rows (normal + attacks)
2. **Normal-only training** — only on normal traffic, so attacks look unusual

The threshold is the 95th percentile of scores on normal training traffic, the same rule used for Mahalanobis and the autoencoder.

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

x_train = df_train.drop(["label", "difficulty"], axis=1)
x_test = df_test.drop(["label", "difficulty"], axis=1)
y_train_binary = (df_train["label"] != "normal").astype(int)
y_test_binary = (df_test["label"] != "normal").astype(int)

# one-hot encode protocol_type, service, flag; test gets the same columns as train
x_train_encoded = pd.get_dummies(x_train)
x_test_encoded = pd.get_dummies(x_test).reindex(columns=x_train_encoded.columns, fill_value=0)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_encoded)
x_test_scaled = scaler.transform(x_test_encoded)

x_normal = x_train_scaled[(y_train_binary == 0).to_numpy()]
print(x_train_scaled.shape, x_test_scaled.shape, x_normal.shape)

(125973, 122) (22544, 122) (67343, 122)


In [4]:
def evaluate(name, iso):
    # score_samples is higher for normal points, so flip it: HIGHER = more likely an attack
    normal_scores = -iso.score_samples(x_normal)
    test_scores = -iso.score_samples(x_test_scaled)

    threshold = np.percentile(normal_scores, 95)  # 5% of normal training traffic gets flagged
    y_pred = (test_scores > threshold).astype(int)

    print(f"--- {name}")
    print("threshold:", threshold)
    print(confusion_matrix(y_test_binary, y_pred))
    print("Precision:", precision_score(y_test_binary, y_pred))
    print("Recall:", recall_score(y_test_binary, y_pred))
    print("F1 Score:", f1_score(y_test_binary, y_pred))
    print("ROC AUC:", roc_auc_score(y_test_binary, test_scores))
    return threshold

In [5]:
# 1. Mixed training: attacks are in the training data, so the forest learns them as "normal"
iso_mixed = IsolationForest(random_state=42).fit(x_train_scaled)
evaluate("Mixed training", iso_mixed)

# sklearn's built-in predict() uses its own cutoff, which flags almost nothing here
y_pred_builtin = (iso_mixed.predict(x_test_scaled) == -1).astype(int)
print("Recall with built-in predict():", recall_score(y_test_binary, y_pred_builtin))

--- Mixed training
threshold: 0.42742822531146457
[[9186  525]
 [9890 2943]]
Precision: 0.8486159169550173
Recall: 0.2293306319644666
F1 Score: 0.36108214219986506
ROC AUC: 0.8652866405309982
Recall with built-in predict(): 7.792410192472531e-05


In [6]:
# 2. Normal-only training
iso = IsolationForest(random_state=42).fit(x_normal)
threshold_iso = evaluate("Normal-only training", iso)

--- Normal-only training
threshold: 0.4835411207880887
[[9068  643]
 [4150 8683]]
Precision: 0.9310529701908643
Recall: 0.6766149770123899
F1 Score: 0.7836996254343608
ROC AUC: 0.9382087710024252


In [7]:
# 3. Normal-only, tuned: more trees and bigger samples per tree
iso_tuned = IsolationForest(n_estimators=300, max_samples=1024, random_state=42).fit(x_normal)
threshold_iso_tuned = evaluate("Normal-only, 300 trees, max_samples=1024", iso_tuned)

--- Normal-only, 300 trees, max_samples=1024
threshold: 0.45641773156554916
[[9046  665]
 [3788 9045]]
Precision: 0.931513903192585
Recall: 0.7048235019091404
F1 Score: 0.8024663975513463
ROC AUC: 0.9479012060726749


### Results on KDDTest+

| Training | Precision | Recall | F1 | ROC AUC |
|---|---|---|---|---|
| Mixed (normal + attacks) | 0.849 | 0.229 | 0.361 | 0.865 |
| Normal-only | 0.931 | 0.677 | 0.784 | 0.938 |
| Normal-only, 300 trees, max_samples=1024 | 0.932 | 0.705 | 0.802 | 0.948 |

- Training on **normal traffic only** triples recall. With attacks in the training data, the forest learns them as part of "normal".
- sklearn's built-in `predict()` on the mixed model flags almost nothing (recall ≈ 0.0001), so a threshold set from normal traffic is needed.
- Isolation Forest still has lower recall than Mahalanobis (0.847) and the autoencoder (0.842).